# CNN vs Transfer Learning untuk Klasifikasi Citra

**Tugas Individu — Mesin Learning 2**
**FarrelGhozy**

Eksperimen perbandingan dua pendekatan:
1. **CNN from scratch** — dataset CIFAR-10 (2 kelas)
2. **Transfer Learning** — dataset Cats vs Dogs dengan pretrained model

---
## Phase 1: Preprocessing & Augmentasi

### 1.1 Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Sequential
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
import time
import random
import zipfile
import urllib.request
import shutil
from pathlib import Path

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Library imported successfully")

---
## A. Dataset 1: CIFAR-10 (CNN from Scratch)

Menggunakan 2 kelas dari CIFAR-10. Pilihan: **cat (3) vs dog (5)**

In [ ]:
# Load CIFAR-10
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

print("CIFAR-10 shape:", x_train_full.shape, x_test_full.shape)

# Pilih 2 kelas: cat (label 3) dan dog (label 5)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
selected_classes = [3, 5]  # cat, dog
selected_labels = ['cat', 'dog']

def filter_classes(x, y, classes):
    mask = np.isin(y, classes).flatten()
    x_filtered = x[mask]
    y_filtered = y[mask]
    y_binary = np.where(y_filtered == classes[0], 0, 1).flatten()
    return x_filtered, y_binary

x_train_cifar, y_train_cifar = filter_classes(x_train_full, y_train_full, selected_classes)
x_test_cifar, y_test_cifar = filter_classes(x_test_full, y_test_full, selected_classes)

print(f"CIFAR-10 (cat vs dog) — Train: {x_train_cifar.shape[0]}, Test: {x_test_cifar.shape[0]}")
print(f"Cat samples: {np.sum(y_train_cifar == 0)}, Dog samples: {np.sum(y_train_cifar == 1)}")

### Visualisasi Sampel CIFAR-10

In [ ]:
def plot_sample_images(x, y, labels, title, rows=2, cols=5):
    fig, axes = plt.subplots(rows, cols, figsize=(12, 5))
    axes = axes.flatten()
    for i in range(rows * cols):
        idx = np.random.randint(len(x))
        axes[i].imshow(x[idx])
        axes[i].set_title(f"{labels[y[idx]]}")
        axes[i].axis('off')
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_sample_images(x_train_cifar, y_train_cifar, selected_labels, "Sample CIFAR-10 (Cat vs Dog)", 2, 5)

---
## B. Dataset 2: Cats vs Dogs (Transfer Learning)

In [ ]:
# Download Cats vs Dogs subset dari TensorFlow dataset
# TFDS menyediakan cats_vs_dogs dalam ukuran manageable

dataset_dir = Path('./dataset/cats_vs_dogs')
dataset_dir.mkdir(parents=True, exist_ok=True)

# Gunakan TensorFlow Datasets atau load dari file yang sudah diunduh
# Jika tidak ada, kita akan buat subset dari data yang ada

try:
    import tensorflow_datasets as tfds
    ds, info = tfds.load('cats_vs_dogs', split='train', with_info=True, as_supervised=True)
    print(f"Total Cats vs Dogs samples: {info.splits['train'].num_examples}")
    
    # Ambil subset 150 per kelas
    cat_count, dog_count = 0, 0
    x_catsvdog, y_catsvdog = [], []
    
    for img, label in ds:
        img = tf.image.resize(img, (128, 128)).numpy().astype(np.uint8)
        if label == 0 and cat_count < 200:
            x_catsvdog.append(img)
            y_catsvdog.append(0)
            cat_count += 1
        elif label == 1 and dog_count < 200:
            x_catsvdog.append(img)
            y_catsvdog.append(1)
            dog_count += 1
        if cat_count >= 200 and dog_count >= 200:
            break
    
    x_catsvdog = np.array(x_catsvdog)
    y_catsvdog = np.array(y_catsvdog)
    print(f"Cats: {cat_count}, Dogs: {dog_count}, Total: {len(x_catsvdog)}")
    
except (ImportError, Exception) as e:
    print(f"TFDS not available or error: {e}")
    print("Generating simulated Cats vs Dogs data from CIFAR-10 cat/dog for structure demonstration...")
    # Fallback: gunakan CIFAR-10 cat/dog untuk struktur notebook
    # Catatan: di run sesungguhnya, ganti dengan dataset asli
    # Kita akan tetap menggunakan subset CIFAR-10 sebagai placeholder
    x_catsvdog = x_train_cifar.copy()
    y_catsvdog = y_train_cifar.copy()
    print("Using CIFAR-10 cat/dog as placeholder — replace with real Cats vs Dogs dataset")

print(f"Cats vs Dogs dataset shape: {x_catsvdog.shape}, labels: {y_catsvdog.shape}")
print(f"Class balance — Cat: {np.sum(y_catsvdog==0)}, Dog: {np.sum(y_catsvdog==1)}")

### Visualisasi Sampel Cats vs Dogs

In [ ]:
plot_sample_images(x_catsvdog, y_catsvdog, ['cat', 'dog'], "Sample Cats vs Dogs Dataset", 2, 5)

---
## C. Data Splitting (70% Train, 15% Val, 15% Test)

In [ ]:
def split_data(x, y, train_ratio=0.7, val_ratio=0.15, random_state=SEED):
    x_temp, x_test, y_temp, y_test = train_test_split(
        x, y, test_size=val_ratio, random_state=random_state, stratify=y
    )
    val_size = val_ratio / (train_ratio + val_ratio)
    x_train, x_val, y_train, y_val = train_test_split(
        x_temp, y_temp, test_size=val_size, random_state=random_state, stratify=y_temp
    )
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

# Split CIFAR-10
(x_train_cifar, y_train_cifar), (x_val_cifar, y_val_cifar), (x_test_cifar, y_test_cifar) = \
    split_data(x_train_cifar, y_train_cifar)
print("=== CIFAR-10 Split ===")
print(f"Train: {len(x_train_cifar)}, Val: {len(x_val_cifar)}, Test: {len(x_test_cifar)}")

# Gabung train+test CIFAR-10 asli untuk dataset lebih besar
# Kita sudah split data train asli (10k) menjadi train/val/test
x_train_catsvdog = x_catsvdog
y_train_catsvdog = y_catsvdog

# Split Cats vs Dogs
(x_train_cvd, y_train_cvd), (x_val_cvd, y_val_cvd), (x_test_cvd, y_test_cvd) = \
    split_data(x_catsvdog, y_catsvdog)
print("\n=== Cats vs Dogs Split ===")
print(f"Train: {len(x_train_cvd)}, Val: {len(x_val_cvd)}, Test: {len(x_test_cvd)}")

---
## D. Preprocessing

### D.1 Normalisasi Pixel Values

In [ ]:
# Normalisasi pixel [0,1]
def normalize(x):
    return x.astype('float32') / 255.0

# CIFAR-10 (32x32)
x_train_cifar_norm = normalize(x_train_cifar)
x_val_cifar_norm = normalize(x_val_cifar)
x_test_cifar_norm = normalize(x_test_cifar)

# Cats vs Dogs (128x128) tetap simpan versi non-normalized untuk visualisasi
x_train_cvd_norm = normalize(x_train_cvd)
x_val_cvd_norm = normalize(x_val_cvd)
x_test_cvd_norm = normalize(x_test_cvd)

print(f"CIFAR-10 — Train: {x_train_cifar_norm.shape}, Range: [{x_train_cifar_norm.min():.2f}, {x_train_cifar_norm.max():.2f}]")
print(f"Cats vs Dogs — Train: {x_train_cvd_norm.shape}, Range: [{x_train_cvd_norm.min():.2f}, {x_train_cvd_norm.max():.2f}]")

### D.2 Resize untuk Transfer Learning

Pretrained model membutuhkan input minimal 224x224 (ResNet50/MobileNetV2).
CIFAR-10 (32x32) cukup untuk CNN from scratch. Cats vs Dogs kita resize.

In [ ]:
IMG_SIZE_TL = (224, 224)  # Input size untuk transfer learning
IMG_SIZE_CNN = (32, 32)   # Input size untuk CNN from scratch (CIFAR-10 native)

# Resize dataset untuk transfer learning
def resize_dataset(x, size):
    x_resized = np.array([
        tf.image.resize(img, size).numpy() for img in x
    ])
    return x_resized

# Note: resize dilakukan nanti di dalam pipeline transfer learning
# untuk efisiensi memori. Sementara simpan ukuran asli.
print(f"CNN input size: {IMG_SIZE_CNN}")
print(f"Transfer Learning input size: {IMG_SIZE_TL}")

### D.3 Augmentasi Data

In [ ]:
# Augmentasi untuk CNN from scratch
data_augmentation_cnn = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

# Augmentasi untuk Transfer Learning
data_augmentation_tl = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

# Demo augmentasi
sample_img = x_train_cifar_norm[0:1]
augmented_images = data_augmentation_cnn(sample_img)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(sample_img[0])
axes[0].set_title("Original")
axes[0].axis('off')
for i in range(1, 4):
    aug = data_augmentation_cnn(sample_img)
    axes[i].imshow(aug[0])
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis('off')
plt.suptitle("Augmentasi Data — CNN from Scratch")
plt.tight_layout()
plt.show()

print("Augmentasi siap digunakan saat training melalui pipeline")

### D.4 Konversi Label ke One-Hot Encoding

In [ ]:
# One-hot encoding untuk categorical crossentropy
y_train_cifar_cat = keras.utils.to_categorical(y_train_cifar, 2)
y_val_cifar_cat = keras.utils.to_categorical(y_val_cifar, 2)
y_test_cifar_cat = keras.utils.to_categorical(y_test_cifar, 2)

y_train_cvd_cat = keras.utils.to_categorical(y_train_cvd, 2)
y_val_cvd_cat = keras.utils.to_categorical(y_val_cvd, 2)
y_test_cvd_cat = keras.utils.to_categorical(y_test_cvd, 2)

print("One-hot encoding selesai")
print(f"CIFAR-10 sample: {y_train_cifar[0]} -> {y_train_cifar_cat[0]}")
print(f"CVD sample: {y_train_cvd[0]} -> {y_train_cvd_cat[0]}")

### Ringkasan Preprocessing

In [ ]:
print("=" * 60)
print("RINGKASAN PREPROCESSING")
print("=" * 60)
print()
print("Dataset CIFAR-10 (CNN from scratch):")
print(f"  - Original size: 32x32x3")
print(f"  - Train: {x_train_cifar.shape[0]} images")
print(f"  - Val:   {x_val_cifar.shape[0]} images")
print(f"  - Test:  {x_test_cifar.shape[0]} images")
print(f"  - Normalization: pixel [0,1]")
print(f"  - Augmentation: Flip, Rotation, Zoom, Translation")
print()
print("Dataset Cats vs Dogs (Transfer Learning):")
print(f"  - Original size: 128x128x3")
print(f"  - Train: {x_train_cvd.shape[0]} images")
print(f"  - Val:   {x_val_cvd.shape[0]} images")
print(f"  - Test:  {x_test_cvd.shape[0]} images")
print(f"  - Normalization: pixel [0,1]")
print(f"  - Resize to: {IMG_SIZE_TL} untuk Transfer Learning")
print(f"  - Augmentation: Flip, Rotation, Zoom, Contrast")

---
## Phase 2: CNN from Scratch

### 2.1 Arsitektur CNN

**Justifikasi Arsitektur:**
- Input: 32x32x3 (ukuran asli CIFAR-10)
- Block 1: Conv2D 32 filter (3x3) + ReLU + MaxPooling (2x2) — ekstraksi fitur low-level
- Block 2: Conv2D 64 filter (3x3) + ReLU + MaxPooling (2x2) — ekstraksi fitur mid-level
- Block 3: Conv2D 128 filter (3x3) + ReLU + MaxPooling (2x2) — ekstraksi fitur high-level
- GlobalAveragePooling2D — mengurangi parameter dibanding Flatten, mencegah overfitting
- Dense 128 + Dropout 0.5 — regularisasi
- Output: Dense 2 + Softmax — klasifikasi biner

**Alasan pemilihan:**
- Filter kecil (3x3) efisien dan bisa menangkap pola lokal
- Jumlah filter bertambah seiring depth (32→64→128) karena fitur semakin kompleks
- GAP mengurangi parameter drastis dibanding Flatten
- Dropout 0.5 mencegah ko-adaptasi neuron

In [ ]:
def build_cnn_from_scratch(input_shape=(32, 32, 3)):
    model = Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Classifier
        layers.GlobalAveragePooling2D(),
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(2, activation='softmax')
    ])
    return model

cnn_model = build_cnn_from_scratch()
cnn_model.summary()

### 2.2 Compile Model

**Justifikasi:**
- Optimizer Adam: adaptive learning rate, cocok untuk CNN
- Learning rate 0.001: default Adam, terbukti stabil
- Loss categorical_crossentropy: untuk klasifikasi multi-class (2 kelas)
- Metrics accuracy: interpretable

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled successfully")

### 2.3 Callbacks

EarlyStopping: hentikan training jika validation loss tidak membaik selama 10 epoch, restore bobot terbaik.
ReduceLROnPlateau: turunkan LR jika loss stagnan.

In [ ]:
callbacks_cnn = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]
print("Callbacks ready")

### 2.4 Training CNN from Scratch

Batch size: 32 — kompromi antara kecepatan dan stabilitas gradient.
Epochs: 50 (dengan EarlyStopping).

In [ ]:
BATCH_SIZE = 32
EPOCHS = 50

# Buat pipeline augmented
def cnn_pipeline(x, y, batch_size, augment=False):
    dataset = tf.data.Dataset.from_tensor_slices((x, y))
    if augment:
        def aug_fn(images, labels):
            return data_augmentation_cnn(images, training=True), labels
        dataset = dataset.map(aug_fn, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds_cnn = cnn_pipeline(x_train_cifar_norm, y_train_cifar_cat, BATCH_SIZE, augment=True)
val_ds_cnn = cnn_pipeline(x_val_cifar_norm, y_val_cifar_cat, BATCH_SIZE, augment=False)

print("Training pipeline ready")
print(f"Training batches: {len(list(train_ds_cnn))}, Val batches: {len(list(val_ds_cnn))}")

In [ ]:
# Record start time
start_time = time.time()

history_cnn = cnn_model.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
    verbose=1
)

training_time_cnn = time.time() - start_time
print(f"\nTraining time: {training_time_cnn:.2f} seconds ({training_time_cnn/60:.2f} minutes)")

### 2.5 Evaluasi CNN from Scratch

In [ ]:
# Evaluasi pada test set
test_ds_cnn = cnn_pipeline(x_test_cifar_norm, y_test_cifar_cat, BATCH_SIZE, augment=False)
test_loss_cnn, test_acc_cnn = cnn_model.evaluate(test_ds_cnn, verbose=0)
val_loss_cnn, val_acc_cnn = cnn_model.evaluate(val_ds_cnn, verbose=0)

train_loss_cnn = history_cnn.history['loss'][-1]
train_acc_cnn = history_cnn.history['accuracy'][-1]

print("=" * 40)
print("CNN FROM SCRATCH — HASIL EVALUASI")
print("=" * 40)
print(f"Train Accuracy: {train_acc_cnn:.4f}")
print(f"Train Loss:     {train_loss_cnn:.4f}")
print(f"Val Accuracy:   {val_acc_cnn:.4f}")
print(f"Val Loss:       {val_loss_cnn:.4f}")
print(f"Test Accuracy:  {test_acc_cnn:.4f}")
print(f"Test Loss:      {test_loss_cnn:.4f}")
print(f"Training Time:  {training_time_cnn:.2f}s")
print(f"Total Params:   {cnn_model.count_params():,}")

### 2.6 Grafik Training CNN from Scratch

In [ ]:
def plot_training_history(history, title_prefix=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Accuracy plot
    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax1.set_title(f'{title_prefix} Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss plot
    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax2.set_title(f'{title_prefix} Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history_cnn, "CNN from Scratch")

### 2.7 Confusion Matrix CNN from Scratch

In [ ]:
# Prediksi
y_pred_cnn = cnn_model.predict(x_test_cifar_norm, verbose=0)
y_pred_cnn_classes = np.argmax(y_pred_cnn, axis=1)

# Confusion Matrix
cm_cnn = confusion_matrix(y_test_cifar, y_pred_cnn_classes)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'])
plt.title('Confusion Matrix — CNN from Scratch')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test_cifar, y_pred_cnn_classes,
                            target_names=['Cat', 'Dog']))

### 2.8 Contoh Prediksi Benar & Salah (CNN from Scratch)

In [ ]:
def plot_predictions(x_test, y_true, y_pred, class_names, title, num_samples=8):
    correct = np.where(y_true == y_pred)[0]
    incorrect = np.where(y_true != y_pred)[0]
    
    # Ambil sample
    correct_samples = np.random.choice(correct, min(num_samples, len(correct)), replace=False)
    incorrect_samples = np.random.choice(incorrect, min(num_samples, len(incorrect)), replace=False)
    
    fig, axes = plt.subplots(2, num_samples, figsize=(16, 4))
    
    for i, idx in enumerate(correct_samples):
        axes[0, i].imshow(x_test[idx])
        axes[0, i].set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9)
        axes[0, i].axis('off')
    axes[0, 0].set_ylabel("Correct", fontsize=12)
    
    for i, idx in enumerate(incorrect_samples):
        axes[1, i].imshow(x_test[idx])
        axes[1, i].set_title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", fontsize=9, color='red')
        axes[1, i].axis('off')
    axes[1, 0].set_ylabel("Incorrect", fontsize=12)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_predictions(x_test_cifar, y_test_cifar, y_pred_cnn_classes,
                 ['Cat', 'Dog'], "CNN from Scratch — Correct vs Incorrect Predictions")

---
## Lanjut ke Phase 3: Transfer Learning